# Pipeline de Poda Estruturada — VGG no CIFAR-10

**Objetivo:** Comparar **Poda Estruturada Local** vs **Poda Estruturada Global** utilizando saliência **L1** e **l2** numa . . .
Este notebook implementa:

1. **Configuração e Importações** — Configuração do dispositivo e monitorização da VRAM.
2. **Poda Estruturada Global** — Classificação e remoção de neurónios em toda a rede com base na sua importância.
3. **Ajuste Fino (*Fine-Tuning*)** — Recuperação iterativa do desempenho após a poda (SGD com Momentum).
4. **Pipeline Automatizado** — Comparação sistemática entre diferentes níveis de esparsidade.
5. **Visualização** — Gráficos comparativos (Precisão, Latência e Taxa de Compressão).


---
## 1. Setup e Importações

Carregamento de todas as dependências, importação das arquiteturas e métricas de `utils.py`, e configuração do dispositivo de computação com tracking inicial de memória VRAM.

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import numpy as np
import random
import copy
import time
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset

# ── Importações do utils.py ──
from utils import (
    CIFAR10MLP,
    evaluate_model,
    compute_metrics,
    train_model,
    classes
)

# ── Reprodutibilidade ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Dispositivo (foco em CUDA) ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    vram_initial = torch.cuda.memory_allocated(device) / (1024**2)
    print(f"GPU: {torch.cuda.get_device_name(device)}")
    print(f"VRAM alocada inicialmente: {vram_initial:.2f} MB")
else:
    print("AVISO: CUDA não disponível. Os resultados de VRAM serão 0.")

Device: cpu
AVISO: CUDA não disponível. Os resultados de VRAM serão 0.


### Carregamento de Dados — Split Determinístico

Para garantir uma comparação justa entre as experiências de poda, reutilizamos o split
determinístico treino/validação gerado durante o treino do baseline.

In [13]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

full_trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

# Reutilizar o split determinístico guardado
split = torch.load("data/cifar10_split.pt", weights_only=False)
train_indices = split["train_indices"]
val_indices = split["val_indices"]

trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

batch_size = 64
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train: {len(trainset):,} | Val: {len(valset):,} | Test: {len(testset):,}")

Train: 42,500 | Val: 7,500 | Test: 10,000


### Carregamento do Modelo Baseline

Carregamento do MLP pré-treinado no CIFAR-10. Este modelo serve como ponto de referência
para todas as experiências de poda subsequentes.

In [ ]:
models_dir = Path('model_dir')
PATH_VGG = models_dir / 'cifar_vgg.pt'

baseline_model = torch.load(PATH_VGG, map_location=torch.device('cpu'), weights_only=False) #MUDA SE QUISERES PASSAR PARA GPU EU É QUE ESTOU COM PROBLEMAS NELA
baseline_model = baseline_model.to(device)
baseline_model.eval()

print(baseline_model)
print(f"\nTotal de parâmetros: {sum(p.numel() for p in baseline_model.parameters()):,}")

if device.type == "cuda":
    vram_model = torch.cuda.memory_allocated(device) / (1024**2)
    print(f"VRAM após carregar modelo: {vram_model:.2f} MB")

CIFAR10MLP(
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3072, out_features=2048, bias=True)
    (2): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.3, inplace=False)
    (5): Linear(in_features=2048, out_features=1024, bias=True)
    (6): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=1024, out_features=512, bias=True)
    (10): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (11): ReLU()
    (12): Dropout(p=0.2, inplace=False)
    (13): Linear(in_features=512, out_features=10, bias=True)
  )
)

Total de parâmetros: 8,928,778


---
## 2. Funções de Saliência para Estimativa de Importância dos Neurónios

A saliência de cada neurónio é calculada com base na norma dos seus pesos.
Neurónios com menor saliência são considerados menos importantes e são candidatos à remoção.

Para o neurónio $j$ na camada com pesos $W$:

**Norma L1:**  $S_j = \sum_i |w_{ij}|$

**Norma L2:**  $S_j = \sqrt{\sum_i w_{ij}^2}$

In [ ]:
def compute_l1_saliency(layer: nn.Module) -> torch.Tensor:
    W = layer.weight.data
    if isinstance(layer, nn.Conv2d):
        # Soma os valores absolutos de todos os pesos do filtro
        return torch.sum(torch.abs(W), dim=(1, 2, 3))
    return torch.sum(torch.abs(W), dim=1)

def compute_l2_saliency(layer: nn.Module) -> torch.Tensor:
    W = layer.weight.data
    if isinstance(layer, nn.Conv2d):
        # Calcula a norma L2 ao longo do filtro completo
        return torch.flatten(W, start_dim=1).norm(p=2, dim=1)
    return torch.norm(W, p=2, dim=1)

def compute_taylor_saliency(model: nn.Module, layer: nn.Module, calibration_loader: DataLoader, device: torch.device) -> torch.Tensor:
    activations = []
    gradients = []
    
    def f_hook(module, input, output): activations.append(output.detach())
    def b_hook(module, grad_in, grad_out): gradients.append(grad_out[0].detach())
    
    h_f = layer.register_forward_hook(f_hook)
    h_b = layer.register_full_backward_hook(b_hook)
    
    model.eval()
    images, labels = next(iter(calibration_loader))
    outputs = model(images.to(device))
    loss = nn.CrossEntropyLoss()(outputs, labels.to(device))
    model.zero_grad()
    loss.backward()
    
    h_f.remove()
    h_b.remove()
    
    act = activations[0]
    grad = gradients[0]
    
    if len(act.shape) == 4: # Se for Conv2d: [Batch, Channels, H, W]
        # Média sobre o Batch (0), Altura (2) e Largura (3) para obter relevância por Canal
        return (act * grad).abs().mean(dim=(0, 2, 3))
    return (act * grad).abs().mean(dim=0)

Funções de saliência definidas: compute_l1_saliency, compute_l2_saliency


### Reconstrução Física das Camadas (In-Place Tensor Manipulation)

Em vez de mascarar pesos a zero, a arquitetura da rede é **fisicamente reconstruída**.
Isto garante que as matrizes de pesos continuam a multiplicar corretamente após a poda,
ajustando dinamicamente `in_features` e `out_features` das camadas subsequentes.

Isto reduz:
- Contagem de parâmetros
- Consumo de memória
- Latência de inferência

In [ ]:
def apply_layer_pruning_physical_vgg(features_sequential: nn.Sequential, layer_idx: int, indices_to_keep: torch.Tensor):
    """
    Efetua a poda estruturada real de filtros numa estrutura sequencial convolucional (VGG.features).
    """
    layer = features_sequential[layer_idx]
    indices_to_keep = torch.sort(indices_to_keep).values
    num_to_keep = len(indices_to_keep)
    
    # 1. Cortar filtros de saída da camada alvo
    layer.weight = nn.Parameter(layer.weight.data[indices_to_keep, :, :, :])
    if layer.bias is not None:
        layer.bias = nn.Parameter(layer.bias.data[indices_to_keep])
    layer.out_channels = num_to_keep
    
    # 2. Propagar alterações para a frente (BatchNorm2d e próxima Conv2d)
    next_conv_idx = None
    for i in range(layer_idx + 1, len(features_sequential)):
        sub_layer = features_sequential[i]
        
        if isinstance(sub_layer, nn.BatchNorm2d):
            sub_layer.weight = nn.Parameter(sub_layer.weight.data[indices_to_keep])
            sub_layer.bias = nn.Parameter(sub_layer.bias.data[indices_to_keep])
            sub_layer.running_mean = sub_layer.running_mean[indices_to_keep]
            sub_layer.running_var = sub_layer.running_var[indices_to_keep]
            sub_layer.num_features = num_to_keep
            
        elif isinstance(sub_layer, nn.Conv2d):
            next_conv_idx = i
            break # Encontrámos a próxima convolução, paramos a propagação sequencial externa
            
    # 3. Ajustar os canais de entrada (dimensão 1 dos pesos) da próxima convolução
    if next_conv_idx is not None:
        next_layer = features_sequential[next_conv_idx]
        next_layer.weight = nn.Parameter(next_layer.weight.data[:, indices_to_keep, :, :])
        next_layer.in_channels = num_to_keep

Função apply_layer_pruning_physical definida.


### Poda Estruturada Local (Local Structured Pruning)

Na poda **local**, cada camada oculta perde exatamente a mesma fração de neurónios
(`prune_ratio`). O ranking é feito **independentemente** por camada.

In [ ]:
# POR FAZER

Função prune_mlp_local definida.


---
## 2. Lógica de Global Structured Pruning

Na poda **global**, a saliência é calculada para **TODOS** os neurónios de **TODAS**
as camadas ocultas, criando um **ranking unificado**. O bottom `prune_ratio` é removido
globalmente — independentemente da camada a que pertençam.

**Vantagem sobre a poda local:**
Preserva mais capacidade nas camadas que são realmente importantes, enquanto
comprime agressivamente camadas com neurónios redundantes.

$$\text{Threshold} = \text{kth\_smallest}\left(\bigcup_{l \in \text{hidden}} \{S_j^{(l)}\}_{j=1}^{n_l},\; k = \lfloor N_{\text{total}} \times r \rfloor\right)$$

In [ ]:
# POR FAZER

Função prune_mlp_global definida.


---
## 3. Ciclo de Fine-Tuning (Iterative Post-Training)

Após o corte in-place dos tensores, os pesos sobreviventes precisam de ser
recalibrados. Este fine-tuning rápido utiliza **SGD com Momentum** (estável
para recalibração pós-poda) durante poucas epochs para recuperar a exatidão.

In [ ]:
# POR FAZER

---
## 4. Pipeline de Avaliação Automatizada

### Função de Avaliação Abrangente

Para cada modelo (baseline ou podado), regista:
- **a)** Exatidão (Accuracy) de teste
- **b)** Contagem de parâmetros ativos
- **c)** Tempo real de inferência (Latência em segundos)
- **d)** Uso de VRAM no dispositivo (se aplicável)

In [ ]:
# POR FAZER

Função evaluate_comprehensive definida.


In [ ]:
import os

save_dir = Path('modelos_podados_vgg')
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# POR FAZER

### Loop de Experimentação

Compara sistematicamente:
- **Tipo de Poda:** Local L1 vs Global L1
- **Sparsity Ratios:** 10%, 20%, 30%, 50%, 70%

Para cada iteração executa: Poda → Fine-Tuning → Avaliação completa.

In [ ]:
# POR FAZER

AVALIAÇÃO DO MODELO BASELINE
  Accuracy:   0.5732
  Parâmetros: 8,928,778
  Latência:   13.5562 s
  VRAM:       0.00 MB

MÉTODO: Local L1 (Magnitude) (one-shot)

───────────────────────────────────────────────────────
  Sparsity Ratio Alvo: 10%
───────────────────────────────────────────────────────
  [One-Shot] A aplicar poda total de 10%...
  Poda Local (ratio=0.10) nas camadas: [1, 5, 9]
    Camada[1]: 2048 → 1843 neurónios (removidos 205)
    Camada[5]: 1024 → 921 neurónios (removidos 103)
    Camada[9]: 512 → 460 neurónios (removidos 52)
    Pré-FT  → Acc: 0.5608 | Params: 7,797,041
  Fine-tuning: 5 epochs | LR=0.0001 | Optimizer=SGD")


    Epoch [1/5]:  63%|██████▎   | 418/665 [00:24<00:10, 24.36it/s, loss=0.784]

### Tabela Resumo dos Resultados

In [ ]:
import pandas as pd
from IPython.display import display

# Preparar a lista de dicionários para o DataFrame
tabela_dados = []

# Extrair métricas do Baseline
baseline_params = baseline_results['param_count']
baseline_lat = baseline_results['latency_seconds']

# Adicionar a linha do Baseline
tabela_dados.append({
    "Método": "Baseline",
    "Sparsity (%)": 0,
    "Accuracy": round(baseline_results['accuracy'], 4),
    "Parâmetros": baseline_params,
    "Compressão (x)": 1.00,
    "Latência (s)": round(baseline_lat, 4),
    "VRAM (MB)": round(baseline_results['vram_mb'], 2)
})

# Iterar sobre os resultados guardados no dicionário all_results
for method_name, results in all_results.items():
    for r in results:
        post = r['post_finetune']
        compression = baseline_params / max(post['param_count'], 1)
        
        tabela_dados.append({
            "Método": method_name,
            "Sparsity (%)": int(r['sparsity_ratio'] * 100),
            "Accuracy": round(post['accuracy'], 4),
            "Parâmetros": post['param_count'],
            "Compressão (x)": round(compression, 2),
            "Latência (s)": round(post['latency_seconds'], 4),
            "VRAM (MB)": round(post['vram_mb'], 2)
        })

# Criar o DataFrame e ordenar para melhor visualização (opcional)
df_resultados = pd.DataFrame(tabela_dados)

# Aplicar um estilo para destacar o Baseline e alinhar o texto
estilo_tabela = df_resultados.style.set_properties(**{'text-align': 'center'}) \
    .set_table_styles([dict(selector='th', props=[('text-align', 'center')])]) \
    .highlight_max(subset=['Accuracy', 'Compressão (x)'], color='lightgreen') \
    .highlight_min(subset=['Latência (s)'], color='lightgreen')

display(estilo_tabela)

NameError: name 'baseline_results' is not defined

---
## 5. Avaliação Global e Visualização

Gráficos comparativos dos resultados experimentais:
1. **Sparsity Ratio vs. Test Accuracy** — Comparação da degradação de exatidão
2. **Sparsity Ratio vs. Inference Latency** — Redução de latência
3. **Compressão de Parâmetros vs. Speedup** — Eficiência da compressão

In [ ]:
# Gráficos


fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle(
    "Análise de Poda Estruturada: Local L1 vs Global L1",
    fontsize=16, fontweight='bold', y=1.02
)

colors = {"Local L1": "#2196F3", "Global L1": "#FF5722"}
markers = {"Local L1": "o", "Global L1": "s"}

baseline_acc = baseline_results['accuracy']
baseline_lat = baseline_results['latency_seconds']
baseline_par = baseline_results['param_count']

# ── Gráfico 1: Sparsity Ratio vs. Test Accuracy ──
ax1 = axes[0]
for method_name, results in all_results.items():
    ratios = [r['sparsity_ratio'] * 100 for r in results]
    accs = [r['post_finetune']['accuracy'] * 100 for r in results]
    ax1.plot(
        ratios, accs,
        marker=markers[method_name],
        color=colors[method_name],
        linewidth=2.5, markersize=9,
        label=method_name
    )

ax1.axhline(
    y=baseline_acc * 100, color='#4CAF50',
    linestyle='--', linewidth=2, alpha=0.8,
    label='Baseline'
)
ax1.set_xlabel('Sparsity Ratio (%)', fontsize=12)
ax1.set_ylabel('Test Accuracy (%)', fontsize=12)
ax1.set_title('Degradação de Exatidão', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10, loc='lower left')
ax1.grid(True, alpha=0.3)
ax1.set_xticks([10, 20, 30, 50, 70])

# ── Gráfico 2: Sparsity Ratio vs. Inference Latency ──
ax2 = axes[1]
for method_name, results in all_results.items():
    ratios = [r['sparsity_ratio'] * 100 for r in results]
    lats = [r['post_finetune']['latency_seconds'] for r in results]
    ax2.plot(
        ratios, lats,
        marker=markers[method_name],
        color=colors[method_name],
        linewidth=2.5, markersize=9,
        label=method_name
    )

ax2.axhline(
    y=baseline_lat, color='#4CAF50',
    linestyle='--', linewidth=2, alpha=0.8,
    label='Baseline'
)
ax2.set_xlabel('Sparsity Ratio (%)', fontsize=12)
ax2.set_ylabel('Latência de Inferência (s)', fontsize=12)
ax2.set_title('Redução de Latência', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks([10, 20, 30, 50, 70])

# ── Gráfico 3: Compressão de Parâmetros vs. Speedup ──
ax3 = axes[2]
for method_name, results in all_results.items():
    compressions = [
        baseline_par / max(r['post_finetune']['param_count'], 1)
        for r in results
    ]
    speedups = [
        baseline_lat / max(r['post_finetune']['latency_seconds'], 1e-9)
        for r in results
    ]
    ax3.plot(
        compressions, speedups,
        marker=markers[method_name],
        color=colors[method_name],
        linewidth=2.5, markersize=9,
        label=method_name
    )

    # Anotar cada ponto com a sparsity ratio
    for i, r in enumerate(results):
        ax3.annotate(
            f"{r['sparsity_ratio']*100:.0f}%",
            (compressions[i], speedups[i]),
            textcoords="offset points",
            xytext=(8, 5), fontsize=8, alpha=0.7
        )

ax3.axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax3.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax3.set_xlabel('Compressão de Parâmetros (x)', fontsize=12)
ax3.set_ylabel('Speedup (x)', fontsize=12)
ax3.set_title('Compressão vs Speedup', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pruning_analysis_vgg.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGráficos guardados em: pruning_analysis_vgg.png")

---
## Conclusão

Este notebook implementou e comparou duas estratégias de **Poda Estruturada** num MLP treinado no CIFAR-10:

| Aspeto | Local L1 | Global L1 |
|--------|----------|-----------|
| **Ranking** | Independente por camada | Unificado (rede inteira) |
| **Distribuição** | Uniforme entre camadas | Adaptativa (baseada em importância real) |
| **Flexibilidade** | Previsível | Mais eficiente em preservar capacidade |

### Próximos Passos
- Integrar saliências baseadas em gradientes (Taylor 1ª ordem, OBD) nos hooks definidos
- Aplicar poda iterativa (múltiplas rondas de poda + fine-tuning)
- Testar com arquiteturas convolucionais (VGG, ResNet)